# 15 OCR layer

Run OCR only on scanned PDFs and image-only files that still have weak or missing extracted text.
This notebook is **opt-in** and **non-destructive**.


In [1]:
from pathlib import Path
from datetime import datetime
import sys
import pandas as pd

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'src').exists() and (candidate / 'notebooks').exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

OUTPUT_DIR = PROJECT_ROOT / 'data' / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('PROJECT_ROOT =', PROJECT_ROOT)


PROJECT_ROOT = C:\00_Developement\sch-file-organizer


In [2]:
from src.ocr_layer import OCRConfig, ensure_ocr_input_schema, identify_ocr_candidates, enrich_with_ocr

def latest_output(prefixes):
    for prefix in prefixes:
        matches = sorted(OUTPUT_DIR.glob(f'{prefix}_*.parquet'))
        if matches:
            return matches[-1]
    return None

INPUT_PATH = latest_output([
    'inventory_with_text',
    'review_snapshot_latest',
    'review_snapshot',
    'inventory',
])

assert INPUT_PATH is not None, 'No upstream parquet found in data/outputs'
print('INPUT_PATH =', INPUT_PATH)


INPUT_PATH = C:\00_Developement\sch-file-organizer\data\outputs\inventory_with_text_smoke_test.parquet


In [3]:
df = pd.read_parquet(INPUT_PATH)
df = ensure_ocr_input_schema(df)
print('Rows:', len(df))
preview_cols = [c for c in ['relative_path', 'suffix', 'text_status', 'text_preview'] if c in df.columns]
display(df[preview_cols].head(10))


Rows: 4


,relative_path,suffix,text_status,text_preview
0,docs/PV15p473-01_PM_PER_environmental-approval...,.pdf,error,
1,docs/Thumbs.db,.db,unsupported,
2,docs/a.txt,.txt,ok,same
3,docs/b.txt,.txt,ok,same


In [4]:
SCAN_ROOT = PROJECT_ROOT / 'data' / 'working' / 'demo_sandbox'
ENABLE_OCR = False
MAX_ROWS = 20
MAX_PDF_PAGES = 3
OCR_LANGUAGES = 'eng'

config = OCRConfig(
    enabled=ENABLE_OCR,
    max_rows=MAX_ROWS,
    max_pdf_pages=MAX_PDF_PAGES,
    languages=OCR_LANGUAGES,
)

candidates = identify_ocr_candidates(df, config)
print('OCR candidates:', len(candidates))
cand_cols = [c for c in ['relative_path', 'suffix', 'text_status', 'text_error'] if c in candidates.columns]
display(candidates[cand_cols].head(50))


OCR candidates: 0


,relative_path,suffix,text_status,text_error


In [5]:
ocr_df = enrich_with_ocr(df, SCAN_ROOT, config)
print('Rows after OCR:', len(ocr_df))
display(
    ocr_df[
        [c for c in [
            'relative_path', 'suffix', 'ocr_candidate', 'ocr_status', 'ocr_backend',
            'ocr_pages_attempted', 'ocr_error', 'ocr_used_for_text'
        ] if c in ocr_df.columns]
    ].head(50)
)


Rows after OCR: 4


,relative_path,suffix,ocr_candidate,ocr_status,ocr_backend,ocr_pages_attempted,ocr_error,ocr_used_for_text
0,docs/PV15p473-01_PM_PER_environmental-approval...,.pdf,False,not_candidate,,0,,False
1,docs/Thumbs.db,.db,False,not_candidate,,0,,False
2,docs/a.txt,.txt,False,not_candidate,,0,,False
3,docs/b.txt,.txt,False,not_candidate,,0,,False


In [6]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
out_csv = OUTPUT_DIR / f'ocr_enriched_{timestamp}.csv'
out_parquet = OUTPUT_DIR / f'ocr_enriched_{timestamp}.parquet'
ocr_df.to_csv(out_csv, index=False, encoding='utf-8-sig')
ocr_df.to_parquet(out_parquet, index=False)
print('Wrote:', out_csv)
print('Wrote:', out_parquet)


Wrote: C:\00_Developement\sch-file-organizer\data\outputs\ocr_enriched_20260307_151618.csv
Wrote: C:\00_Developement\sch-file-organizer\data\outputs\ocr_enriched_20260307_151618.parquet


In [7]:
display(
    ocr_df.loc[
        ocr_df['ocr_status'].fillna('').astype(str).isin(['success', 'error', 'empty']),
        [c for c in [
            'relative_path', 'ocr_status', 'ocr_backend', 'ocr_error',
            'ocr_text_preview', 'text_preview_after_ocr'
        ] if c in ocr_df.columns]
    ].head(50)
)


,relative_path,ocr_status,ocr_backend,ocr_error,ocr_text_preview,text_preview_after_ocr
